# Crypto intraday data-quality audit — 2026-08-15

## tl;dr

All 26 commissioned crypto H1/M5 lanes were inspected over their latest 270 completed bars. After correction, 22 lanes are fully usable and four are usable for price structure but have no volume. PEPE H1/M5 were repaired with CoinGecko's verified PEPE/WETH pool after both CoinGecko and CoinMarketCap were tested. BTC, ETH, and SOL remain on their approved liquid Binance USDT representations. Kevin's DEX Screener evidence is useful as an independent reference-price check, but it is not historical OHLC and therefore is not a valid Fragarach candle provider.

## Context & Methods

The decision is whether Kevin's provider or another reviewed source should replace weak Fragarach crypto H1/M5 data and which current lanes are safe for Gator's 30×30×30 review. The audit checks OHLC geometry, interval continuity, zero-range candles, zero-volume coverage, close-price resolution, and H1 boundary discontinuity.

### Key Assumptions

- Price chartability and volume availability are classified separately.
- A lane needs at least 30 completed bars for review.
- DEX Screener rolling M5/H1 activity snapshots do not constitute historical candles.
- Provider comparison requests are read only; canonical correction is performed by Fragarach's governed ingestion path outside this notebook.

In [1]:
from pathlib import Path
import json, sqlite3, sys

repo = Path.cwd()
sys.path.insert(0, str(repo / 'src'))
authority_db = repo / 'data/runtime/spec002_real_evidence_acceptance.sqlite3'
kevin_db = Path('/Users/raymorgan/VSC/Kevin/data/imported/dex_intelligence.sqlite')
assert authority_db.exists() and kevin_db.exists()
print('Inputs located:', authority_db.name, kevin_db.name)

Inputs located: spec002_real_evidence_acceptance.sqlite3 dex_intelligence.sqlite


## Data

### 1. Run the Fragarach estate check

In [2]:
from fragarach_ii.crypto_intraday_quality import audit_crypto_intraday_quality

audit = audit_crypto_intraday_quality(authority_db, window=270)
print('Lane count:', audit['lane_count'])
print('Status counts:', json.dumps(audit['status_counts'], sort_keys=True))

Lane count: 26
Status counts: {"USABLE": 22, "USABLE_PRICE_ONLY": 4}


### 2. Confirm Kevin coverage and data shape

In [3]:
with sqlite3.connect(kevin_db) as connection:
    tables = {row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    coverage = connection.execute(
        """SELECT upper(symbol), group_concat(DISTINCT chain_id), count(*),
                  min(captured_at), max(captured_at)
             FROM token_aggregates
            WHERE upper(symbol) IN ('ADA','AVAX','BNB','BTC','DOGE','DOT','ETH','HYPE','LINK','PEPE','SOL','SUI','XRP')
            GROUP BY upper(symbol) ORDER BY upper(symbol)"""
    ).fetchall()
print('OHLC/candle tables present:', sorted(name for name in tables if 'ohlc' in name.lower() or 'candle' in name.lower()))
print('Overlapping Kevin symbols:')
for row in coverage:
    print('  ', row)

OHLC/candle tables present: []
Overlapping Kevin symbols:
   ('BNB', 'ethereum', 1692, '2026-08-08T13:39:16.429625+00:00', '2026-08-15T12:34:58.123750+00:00')
   ('LINK', 'bsc', 2844, '2026-08-01T13:19:18.974029+00:00', '2026-08-15T12:34:58.123750+00:00')
   ('PEPE', 'bsc', 2894, '2026-08-01T11:50:12.179606+00:00', '2026-08-15T12:34:58.123750+00:00')
   ('SUI', 'sui', 1772, '2026-08-08T06:46:17.319973+00:00', '2026-08-15T12:34:58.123750+00:00')


### 3. Compare PEPE provider candidates over 270 completed intervals

CoinGecko is tested on the verified Ethereum PEPE/WETH pool with empty DEX intervals retained. CoinMarketCap is tested on its public PEPE token K-line endpoint; skipped M5 intervals are continuity-filled using the previous close and zero volume before scoring.

In [4]:
import urllib.request
from datetime import UTC, datetime
from fragarach_ii.crypto_intraday_quality import assess_crypto_intraday_rows

TOKEN = '0x6982508145454ce325ddbe47a25d4ec3d2311933'
POOL = '0xa43fe16908251ee70ef74718545e4fe6c5ccec9f'
now = datetime.now(UTC).timestamp()

def fetch_json(url):
    request = urllib.request.Request(url, headers={'User-Agent': 'FragarachDataQualityAudit/1.0', 'Accept': 'application/json'})
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)

def completed(rows, step):
    return sorted(row for row in rows if row[0] + step <= now)

def continuity_fill(rows, step):
    by_time = {row[0]: row for row in rows}
    result, previous_close = [], None
    for timestamp in range(rows[0][0], rows[-1][0] + 1, step):
        row = by_time.get(timestamp)
        if row is None and previous_close is not None:
            row = (timestamp, previous_close, previous_close, previous_close, previous_close, 0.0)
        if row is not None:
            result.append(row); previous_close = row[4]
    return result

comparison = []
for timeframe, path, aggregate, step, cmc_interval in [('H1','hour',1,3600,'1h'), ('M5','minute',5,300,'5min')]:
    cg = fetch_json(f'https://api.geckoterminal.com/api/v2/networks/eth/pools/{POOL}/ohlcv/{path}?token=base&aggregate={aggregate}&limit=1000&currency=usd&include_empty_intervals=true')
    cg_rows = [(int(row[0]), *map(float, row[1:6])) for row in cg['data']['attributes']['ohlcv_list']]
    cg_rows = completed(cg_rows, step)[-270:]
    cmc = fetch_json(f'https://pro-api.coinmarketcap.com/public-api/v1/k-line/candles?platform=ethereum&address={TOKEN}&interval={cmc_interval}&unit=usd&limit=1000&pm=p')
    cmc_rows = []
    for row in cmc['data']:
        timestamp = int(float(row[5]) / 1000 if float(row[5]) > 10_000_000_000 else float(row[5]))
        cmc_rows.append((timestamp, *map(float, row[:5])))
    cmc_rows = continuity_fill(completed(cmc_rows, step), step)[-270:]
    for source, rows in [('CoinGecko DEX', cg_rows), ('CoinMarketCap', cmc_rows)]:
        result = assess_crypto_intraday_rows(asset='PEPEUSD', timeframe=timeframe, rows=rows, provider=source, provider_symbol=TOKEN, requested_window=270)
        comparison.append((source, timeframe, result['status'], result['metrics']['distinct_close_ratio'], result['metrics']['zero_range_ratio'], result['metrics']['missing_interval_count'], result['metrics']['invalid_ohlc_count']))
for row in comparison:
    print(f'{row[0]:16} {row[1]:2} {row[2]:8} distinct={row[3]:.1%} zero_range={row[4]:.1%} missing={row[5]} invalid={row[6]}')

CoinGecko DEX    H1 USABLE   distinct=99.6% zero_range=0.4% missing=0 invalid=0
CoinMarketCap    H1 UNUSABLE distinct=99.6% zero_range=0.4% missing=0 invalid=1
CoinGecko DEX    M5 USABLE   distinct=58.1% zero_range=42.2% missing=0 invalid=0
CoinMarketCap    M5 USABLE   distinct=80.4% zero_range=20.0% missing=0 invalid=0


## Results

### 4. Inspect every non-standard lane outcome

In [5]:
for lane in audit['lanes']:
    if lane['status'] != 'USABLE':
        reasons = ', '.join(lane['reason_codes'])
        print(f"{lane['lane']:12} {lane['status']:18} {lane['provider'] or '-':12} {lane['provider_symbol'] or '-':12} {reasons}")

HYPEUSD:H1   USABLE_PRICE_ONLY  TWELVE_DATA  HYPE/USDT    VOLUME_UNAVAILABLE
HYPEUSD:M5   USABLE_PRICE_ONLY  TWELVE_DATA  HYPE/USDT    VOLUME_UNAVAILABLE
SUIUSD:H1    USABLE_PRICE_ONLY  TWELVE_DATA  SUI/USD      VOLUME_UNAVAILABLE
SUIUSD:M5    USABLE_PRICE_ONLY  TWELVE_DATA  SUI/USD      VOLUME_UNAVAILABLE


## Takeaways

1. Kevin is not the cause of the H1/M5 chart errors and cannot replace a candle provider: it stores immutable DEX snapshots and rolling activity windows, not historical OHLC.
2. The main defect was provider-market selection. BTCUSD, ETHUSD, and SOLUSD are now backed by their approved liquid Binance USDT representations.
3. CoinGecko's verified PEPE/WETH pool is materially better than Twelve Data and Binance for PEPE H1/M5. CoinMarketCap improved M5 but its longer H1 sample contained one invalid candle.
4. PEPE H1/M5 now pass with complete 270-bar windows and are available to Gator; they were not retired.
5. HYPE and SUI remain usable for Gator price-structure work, with an explicit warning that volume is unavailable. Previously quarantined bad-data reviews stay quarantined.